# TODO: You must set checkpoint_directory to where you want to save and load the model

In [1]:
checkpoint_base_directory ="/Users/jameswalker/Programming/Checkpoints5/MappoCheckpoint"
checkpoint_number=0

In [2]:
import ray 

ray.shutdown()
ray.init() 

2025-04-27 21:28:13,422	INFO worker.py:1852 -- Started a local Ray instance.


Python version:,3.9.17
Ray version:,2.44.1


In [3]:
from ray.rllib.algorithms.ppo import PPOConfig
from ray.rllib.core.rl_module.default_model_config import DefaultModelConfig
from tqdm import tqdm 
from metadrive.envs.marl_envs.roundabout_rllib_delegator_env import RoundaboutRLLibDelegatorEnv 
from ray.rllib.core.rl_module.multi_rl_module import MultiRLModuleSpec
from ray.rllib.core.rl_module.rl_module import RLModuleSpec

config = ( PPOConfig()
    .environment(
        RoundaboutRLLibDelegatorEnv,
    )
    .env_runners(
        num_env_runners=1, 
        num_cpus_per_env_runner=1,
        num_gpus_per_env_runner=0,
        # I don't think there's any use for this since the observations are already 1D ndarrays; It comes from their Multi-Agent Rock Paper Scissors example
        # env_to_module_connector=lambda env: FlattenObservations(multi_agent=True),
    )
    # .env_runners(
    #     num_env_runners=3, 
    #     num_cpus_per_env_runner=1,
    #     num_gpus_per_env_runner=0
    #     )
    .multi_agent(
        policies={"agent0", "agent1"},
        policy_mapping_fn=lambda aid, *_: aid,   # one policy per agent
    )
    .framework('torch')
    .learners(
        num_learners=1,
        num_cpus_per_learner=1,
    )
    # .learners(
    #     num_learners=2,
    #     num_cpus_per_learner=2,
    #     # num_gpus_per_learner=0,
    # )
    .training(lr=3e-4)                          # PPO default
    .rl_module(
        model_config=DefaultModelConfig(
                use_lstm=False,
                fcnet_hiddens=[128, 64], # Two hidden layers 
                fcnet_activation="relu",
                vf_share_layers=True,
            ),
        rl_module_spec=MultiRLModuleSpec(
            rl_module_specs={
                "agent0": RLModuleSpec(),
                "agent1": RLModuleSpec(),
            }
        ),
    )
    # .evaluation(
    #     # evaluation_interval=1,
    #     evaluation_num_env_runners=1, 
    #     # evaluation_duration=10,
    # )
    .debugging(log_level='INFO') # INFO, DEBUG, ERROR, WARN
)



In [4]:
algo = config.build_algo()


2025-04-27 21:28:39,028	WARNING algorithm_config.py:4704 -- You are running PPO on the new API stack! This is the new default behavior for this algorithm. If you don't want to use the new API stack, set `config.api_stack(enable_rl_module_and_learner=False,enable_env_runner_and_connector_v2=False)`. For a detailed migration guide, see here: https://docs.ray.io/en/master/rllib/new-api-stack-migration-guide.html
/Users/jameswalker/miniconda3/lib/python3.9/site-packages/ray/rllib/algorithms/algorithm.py:512: RayDeprecationWarning: This API is deprecated and may be removed in future Ray releases. You could suppress this warning by setting env variable PYTHONWARNINGS="ignore::DeprecationWarning"
`UnifiedLogger` will be removed in Ray 2.7.
  return UnifiedLogger(config, logdir, loggers=None)
/Users/jameswalker/miniconda3/lib/python3.9/site-packages/ray/tune/logger/unified.py:53: RayDeprecationWarning: This API is deprecated and may be removed in future Ray releases. You could suppress this wa

(MultiAgentEnvRunner pid=69789) Overall episode reward 0
(MultiAgentEnvRunner pid=69789) Overall episode driving reward 0
(MultiAgentEnvRunner pid=69789) Overall episode speed reward 0
Overall episode reward 0
Overall episode driving reward 0
Overall episode speed reward 0


2025-04-27 21:28:42,796	INFO connector_pipeline_v2.py:272 -- Added AddObservationsFromEpisodesToBatch to the end of EnvToModulePipeline.
2025-04-27 21:28:42,804	INFO connector_pipeline_v2.py:272 -- Added AddTimeDimToBatchAndZeroPad to the end of EnvToModulePipeline.
2025-04-27 21:28:42,840	INFO connector_pipeline_v2.py:272 -- Added AddStatesFromEpisodesToBatch to the end of EnvToModulePipeline.
2025-04-27 21:28:42,921	INFO connector_pipeline_v2.py:272 -- Added AgentToModuleMapping to the end of EnvToModulePipeline.
2025-04-27 21:28:42,932	INFO connector_pipeline_v2.py:272 -- Added BatchIndividualItems to the end of EnvToModulePipeline.
2025-04-27 21:28:42,944	INFO connector_pipeline_v2.py:272 -- Added NumpyToTensor to the end of EnvToModulePipeline.
2025-04-27 21:28:42,947	WARNING deprecation.py:50 -- DeprecationWarning: `RLModule(config=[RLModuleConfig object])` has been deprecated. Use `RLModule(observation_space=.., action_space=.., inference_only=.., model_config=.., catalog_class=

(_WrappedExecutable pid=69790) Setting up process group for: env:// [rank=0, world_size=1]
(_WrappedExecutable pid=69790) 2025-04-27 21:28:47,723	WARNING deprecation.py:50 -- DeprecationWarning: `RLModule(config=[RLModuleConfig object])` has been deprecated. Use `RLModule(observation_space=.., action_space=.., inference_only=.., model_config=.., catalog_class=..)` instead. This will raise an error in the future!


# TRAIN THE ALGORITHM

In [5]:
# This trains the algorithm
# Prints metrics every few iterations
# Saves/checkpoints the algorithm to your disk in a new folder eachtime

iteration_num = 1_000
print_every_num_iterations = 1
save_every_num_iterations = 50
for iteration in tqdm(range(0, iteration_num)):
    results = algo.train()
    if iteration % print_every_num_iterations == 0:
        # Print
        if "env_runners" in results:
            mean_return = results["env_runners"].get("episode_return_mean", float("nan"))
            print(f"Iteration={iteration} Episode Mean Return={mean_return}")

        # Save if multiple of save_every_num_iterations
        if iteration % save_every_num_iterations == 0:
            # Save Algorithm checkpoint.
            checkpoint_directory = checkpoint_base_directory + str(checkpoint_number)
            algo.save_to_path(checkpoint_directory)
            checkpoint_number = checkpoint_number + 1
            print(f"Saved to {checkpoint_directory}")


  0%|          | 0/1000 [00:00<?, ?it/s]

(MultiAgentEnvRunner pid=69789) Overall episode reward 0
(MultiAgentEnvRunner pid=69789) Overall episode driving reward 0
(MultiAgentEnvRunner pid=69789) Overall episode speed reward 0
(MultiAgentEnvRunner pid=69789) Overall episode reward -5390.837829917053
(MultiAgentEnvRunner pid=69789) Overall episode driving reward 23.132665998048616
(MultiAgentEnvRunner pid=69789) Overall episode speed reward 1.8856863473465353
(MultiAgentEnvRunner pid=69789) Overall episode reward -7904.935398767497
(MultiAgentEnvRunner pid=69789) Overall episode driving reward 22.01358103219404
(MultiAgentEnvRunner pid=69789) Overall episode speed reward 1.921291745711901
(MultiAgentEnvRunner pid=69789) Overall episode reward -8163.6516908777185
(MultiAgentEnvRunner pid=69789) Overall episode driving reward 22.086096034164868
(MultiAgentEnvRunner pid=69789) Overall episode speed reward 1.8960071480570093


  0%|          | 1/1000 [00:14<4:05:13, 14.73s/it]

Iteration=0 Episode Mean Return=-5359.669970646398
Saved to /Users/jameswalker/Programming/Checkpoints5/MappoCheckpoint0
(MultiAgentEnvRunner pid=69789) Overall episode reward 20.745032439276432
(MultiAgentEnvRunner pid=69789) Overall episode driving reward 19.000894433088305
(MultiAgentEnvRunner pid=69789) Overall episode speed reward 1.7441380061881215
(MultiAgentEnvRunner pid=69789) Overall episode reward 25.13550728152425
(MultiAgentEnvRunner pid=69789) Overall episode driving reward 23.19649124006081
(MultiAgentEnvRunner pid=69789) Overall episode speed reward 1.9390160414634894
(MultiAgentEnvRunner pid=69789) Overall episode reward -9828.979443467219
(MultiAgentEnvRunner pid=69789) Overall episode driving reward 23.613759403832894
(MultiAgentEnvRunner pid=69789) Overall episode speed reward 1.942833107171788
(MultiAgentEnvRunner pid=69789) Overall episode reward 22.412161191091435
(MultiAgentEnvRunner pid=69789) Overall episode driving reward 20.574084554045157
(MultiAgentEnvRunn

  0%|          | 2/1000 [00:29<4:02:40, 14.59s/it]

Iteration=1 Episode Mean Return=-5247.127465524551
(MultiAgentEnvRunner pid=69789) Overall episode reward -10306.738041956394
(MultiAgentEnvRunner pid=69789) Overall episode driving reward 23.468951902961507
(MultiAgentEnvRunner pid=69789) Overall episode speed reward 1.9314130345881346
(MultiAgentEnvRunner pid=69789) Overall episode reward -7535.46719160338
(MultiAgentEnvRunner pid=69789) Overall episode driving reward 20.181092778292623
(MultiAgentEnvRunner pid=69789) Overall episode speed reward 1.758049962422735
(MultiAgentEnvRunner pid=69789) Overall episode reward -1881.5635140338816
(MultiAgentEnvRunner pid=69789) Overall episode driving reward 18.477659823193065
(MultiAgentEnvRunner pid=69789) Overall episode speed reward 1.7572146392645835
(MultiAgentEnvRunner pid=69789) Overall episode reward -8069.760742235354
(MultiAgentEnvRunner pid=69789) Overall episode driving reward 15.72717144271003
(MultiAgentEnvRunner pid=69789) Overall episode speed reward 1.5556813555132556


  0%|          | 2/1000 [00:43<6:01:28, 21.73s/it]


KeyboardInterrupt: 

# Save Algorithm instance
"An Algorithm instance typically holds a neural network for computing actions, called the policy, the RL environment you want to optimize against, a loss function, an optimizer, and some code describing the algorithm’s execution logic, like determining when to collect samples, when to update your model, etc."

We can checkpoint an Algorithm to save its state then load it into a new Algorithm instance at a later date.

In [6]:
# Save Algorithm checkpoint.
algo.save_to_path(checkpoint_directory)

# Display the Algorithm RLModule to check we loaded the same RLModule later on
print(type(algo))
algo.get_module()

<class 'ray.rllib.algorithms.ppo.ppo.PPO'>


# How to debug AssertionError
AssertionError: Can not call this API after engine initialization!

Run the cell below. It should fix this error. This error gets raised if you press 'ESC' and shutdown the Environment but the renderer doesn't properly shutdown.

In [3]:
env.close()